In [ ]:
# Run this entire file as ONE Kaggle notebook cell (GPU + Internet enabled).
# The optional path overrides are the only settings normally needing edits.
SENSOR = "QB"
K = 4
EPOCHS = 200
HOLD_LR_EPOCHS = 100  # 1..100 fixed LR; 101..200 cosine decay.
MIN_LEARNING_RATE = 1e-6
STATUS_INTERVAL_SECONDS = 15
CHECKPOINT_INTERVAL_SECONDS = 60
FUSED_ADAM = True  # Same optimizer implementation in both arms; GPU AMP overflow handling.
USE_FFT_OBSERVATION = False  # Same FIR/phase/boundary; GPU checks forward AND backward first.
RESUME_INPUT = ""  # Saved Output directory from THIS cell; auto-find a unique matching output.
# Default: fresh B1/B1_AA pair. Existing legacy 100-epoch weights are not imported silently.
SEEDS = [42]
VARIANTS = ["B1", "B1_AA"]
EFFECTIVE_BATCH_SIZE = 32
LEARNING_RATE = 1e-4
LEGACY_MICRO_BATCH_SIZE = 32  # One forward/backward per effective batch; equal in B1/B1_AA.
# If either arm fails the preflight with OOM, set this to 16 in BOTH arms.
# Different micro batches change floating-point accumulation, although batch/loss remain equal.
TRAINING_MODES = ["B1", "B1_AA"]

SWAP_GPUS = False  # Swap the two physical GPUs for a repeat in a new output directory.
CONTROLLED_COMPARISON = True  # Same micro batch in both arms; False benchmarks it per model.
AMP = True
USE_ALL_GPUS = True
GPU_CACHE = True
CHANNELS_LAST = True
NUM_WORKERS = 0  # No CPU DataLoader in GPU-resident training.
SAVE_PREDICTIONS = True
DATA_ROOT = "/kaggle/input/datasets/biyonghiyon/ctrs-pan-sharpening-dataset"

TRAIN_H5 = f"{DATA_ROOT}/QuickBird__Training_Dataset__train_qb.h5"
VAL_H5   = f"{DATA_ROOT}/QuickBird__Training_Dataset__valid_qb.h5"
RR_H5    = f"{DATA_ROOT}/QuickBird__Testing_Dataset_ReducedData_H5_Format__test_qb_multiExm1.h5"
FR_H5    = f"{DATA_ROOT}/QuickBird__Testing_Dataset_FullData_H5_Format__test_qb_OrigScale_multiExm1.h5"
OUTPUT_DIR = "/kaggle/working/ssa_kaggle_bandlimit_k4_v1_fix_20261010"
REPO_COMMIT = "be02b7cbf03fe00681194ae9fc9419661f24cd56"
UPSTREAM_COMMIT = "a4ca40e407b12bf4c30f804384405ce321d11c51"
PLAN_COMMIT = None  # Architecture is defined directly in this cell.

print("[FIXED VERSION v2] Safe model-code mismatch handling enabled; fresh OUTPUT_DIR selected.", flush=True)
print("[PERSISTENCE] Enable Files persistence for interactive sessions. Save a Version with Output and verify/download best/latest before leaving. Unsaved files deleted by Kaggle cannot be recovered by this cell.",flush=True)
print("[START] B1/B1_AA: 200 epochs, fixed LR 100 + cosine 100, GPU-resident pair", flush=True)
import hashlib
import ast
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile

for module, package in [("h5py", "h5py"), ("scipy", "scipy"),
                        ("skimage", "scikit-image"), ("PIL", "pillow"),
                        ("matplotlib", "matplotlib"), ("psutil", "psutil")]:
    if importlib.util.find_spec(module) is None:
        print(f"[SETUP] Installing {package}", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True, timeout=180)
import h5py
import numpy as np
import torch
from IPython.display import clear_output, display, FileLink

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required. Enable a Kaggle GPU accelerator.")
print(f"[CUDA] {torch.cuda.device_count()} GPUs detected", flush=True)
suite_started = time.monotonic()
if not (0 < HOLD_LR_EPOCHS < EPOCHS and 0 < MIN_LEARNING_RATE <= LEARNING_RATE):
    raise ValueError("Require 0 < HOLD_LR_EPOCHS < EPOCHS, 0 < minimum LR <= initial LR")
if not (STATUS_INTERVAL_SECONDS >= 5 and CHECKPOINT_INTERVAL_SECONDS >= 60):
    raise ValueError("Progress/checkpoint intervals are too small")
CELL_ID = "ssa_bandlimit_k4_v1"
suite_settings = {"cell_id": CELL_ID, "repo_commit": REPO_COMMIT,
    "upstream_commit": UPSTREAM_COMMIT, "sensor": SENSOR, "k": K, "epochs": EPOCHS,
    "seeds": SEEDS, "variants": VARIANTS, "batch": EFFECTIVE_BATCH_SIZE,
    "micro": LEGACY_MICRO_BATCH_SIZE, "lr": LEARNING_RATE, "hold": HOLD_LR_EPOCHS,
    "min_lr": MIN_LEARNING_RATE, "amp": AMP, "channels_last": CHANNELS_LAST,
    "fused_adam": FUSED_ADAM, "fft_observation": USE_FFT_OBSERVATION,
    "controlled": CONTROLLED_COMPARISON, "save_predictions": SAVE_PREDICTIONS,
    "status_seconds": STATUS_INTERVAL_SECONDS, "checkpoint_seconds": CHECKPOINT_INTERVAL_SECONDS}
if SENSOR != "QB":
    raise ValueError("This audited first experiment is QB only.")
if VARIANTS != ["B1", "B1_AA"] or TRAINING_MODES != ["B1", "B1_AA"]:
    raise ValueError("B1 and B1_AA only; B0 is excluded.")
if len(set(SEEDS)) != len(SEEDS) or EPOCHS < 1 or EFFECTIVE_BATCH_SIZE < 1 or LEARNING_RATE <= 0:
    raise ValueError("Invalid run settings")
if not GPU_CACHE or not CONTROLLED_COMPARISON or LEGACY_MICRO_BATCH_SIZE < 1:
    raise ValueError("GPU residency and a shared fixed micro batch are required")
if not USE_ALL_GPUS:
    raise ValueError("USE_ALL_GPUS must remain True for parallel B1/B1_AA")
if torch.cuda.device_count() < 2:
    raise RuntimeError("Select Kaggle T4 x2: B1 and B1_AA require two GPUs for parallel execution")
SCALE = {"QB": 2047., "GF2": 1023., "WV3": 2047.}[SENSOR]
OUT = Path(OUTPUT_DIR)
# Restore only this experiment's exact settings, never legacy or unrelated runs.
if not (OUT / "suite_config.json").exists():
    candidates = ([Path(RESUME_INPUT) / "suite_config.json"] if RESUME_INPUT else
                  sorted(Path("/kaggle/input").rglob("suite_config.json")))
    matches = []
    for candidate in candidates:
        if candidate.is_file():
            try:
                if json.loads(candidate.read_text()) == suite_settings:
                    matches.append(candidate.parent)
            except (OSError, ValueError):
                pass
    if RESUME_INPUT and len(matches) != 1:
        raise ValueError("RESUME_INPUT does not contain this cell's matching suite_config.json")
    if len(matches) > 1:
        raise ValueError(f"Multiple matching previous Outputs; set RESUME_INPUT: {matches}")
    if matches:
        if OUT.exists() and any(OUT.iterdir()):
            raise ValueError("Nonempty output without suite_config.json; choose a new OUTPUT_DIR")
        print(f"[RESTORE] {matches[0]} -> {OUT}", flush=True)
        shutil.copytree(matches[0], OUT, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.tmp", "*.pyc", "__pycache__"))
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / "suite_config.json").exists():
    if json.loads((OUT / "suite_config.json").read_text()) != suite_settings:
        raise ValueError("Experiment settings changed; choose a new OUTPUT_DIR")


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def sha(path):
    digest = hashlib.sha256()
    path = Path(path)
    size = path.stat().st_size; done = 0; started = last = time.monotonic()
    if size >= 64*1024**2: print(f"[HASH] {path.name}: {size/1024**2:.0f} MB", flush=True)
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block); done += len(block)
            if time.monotonic()-last >= 5:
                print(f"[HASH] {path.name}: {100*done/max(size,1):.1f}% | {done/1024**2:.0f}/{size/1024**2:.0f} MB | {time.monotonic()-started:.0f}s", flush=True)
                last = time.monotonic()
    if size >= 64*1024**2: print(f"[HASH DONE] {path.name}: {time.monotonic()-started:.1f}s", flush=True)
    return digest.hexdigest()


def locate(override, filename):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Explicit H5 path is missing: {path}")
        if "._" in path.name or not h5py.is_hdf5(path):
            raise ValueError(f"AppleDouble/non-HDF5 input: {path}")
        return path.resolve()
    # Dataset files carry long QuickBird__... prefixes; skip 0 MB AppleDouble files.
    candidates = sorted(p for p in Path(DATA_ROOT).rglob("*.h5")
                        if (p.name == filename or p.name.endswith("__" + filename))
                        and "._" not in p.name and p.stat().st_size > 0 and h5py.is_hdf5(p))
    if len(candidates) != 1:
        raise RuntimeError(f"Set the explicit path for {filename}; valid candidates: {candidates}")
    return candidates[0].resolve()


write_json(OUT / "suite_config.json", suite_settings)


stem = SENSOR.lower()
PATHS = {"train": locate(TRAIN_H5, f"train_{stem}.h5"),
         "val": locate(VAL_H5, f"valid_{stem}.h5"),
         "RR": locate(RR_H5, f"test_{stem}_multiExm1.h5"),
         "FR": locate(FR_H5, f"test_{stem}_OrigScale_multiExm1.h5")}
if len(set(PATHS.values())) != 4:
    raise ValueError("Train, validation, RR and FR must be distinct files.")

# Fetch just the pinned source files, not the repository's large checkpoints.
SOURCE = OUT / "code" / "CtrS"
custom_path = OUT / "code" / "B1_AA_models.py"
custom_hash = None
PASTED_MODEL_SOURCE = r'''
import torch
from torch import nn
from torch.nn import functional as F
from ssamrn.models.ssa_mrn import RestoredPansharpeningNet
from ssamrn.observation import SensorObservation

class FFTObservation(SensorObservation):
    def __init__(self, sensor):
        super().__init__(sensor)
        self.fft_enabled = False
        self._geometry = {}

    def blur(self, prediction):
        if not self.fft_enabled:
            return super().blur(prediction)
        if prediction.ndim != 4 or prediction.shape[1] != self.kernel.shape[0]:
            raise ValueError("Expected sensor-band NCHW")
        h, w = prediction.shape[-2:]
        if h % 4 or w % 4:
            raise ValueError("Observation dimensions must be divisible by four")
        key = (h, w, prediction.device, prediction.dtype)
        if key not in self._geometry:
            from scipy.fft import next_fast_len
            kh, kw = self.kernel.shape[-2:]
            ph, pw = kh // 2, kw // 2
            rows = torch.arange(-ph, h + ph, device=prediction.device).clamp(0, h - 1)
            cols = torch.arange(-pw, w + pw, device=prediction.device).clamp(0, w - 1)
            shape = (next_fast_len(h + 2*ph + kh - 1), next_fast_len(w + 2*pw + kw - 1))
            # conv2d is correlation: reverse the FIR for FFT linear convolution.
            kernel = self.kernel[:, 0].to(dtype=prediction.dtype).flip((-2, -1))
            frequency = torch.fft.rfft2(kernel, s=shape)
            self._geometry[key] = (rows, cols, shape, frequency, kh, kw)
        rows, cols, shape, frequency, kh, kw = self._geometry[key]
        padded = prediction.index_select(-2, rows).index_select(-1, cols)
        full = torch.fft.irfft2(torch.fft.rfft2(padded, s=shape) * frequency, s=shape)
        return full[..., kh-1:kh-1+h, kw-1:kw-1+w]


class BandLimitedDownsample(nn.Module):
    """Positive unit-sum FIR then endpoint-aligned sampling, fixed QB hypothesis."""
    def __init__(self, ratio):
        super().__init__()
        self.ratio = ratio

    def forward(self, x):
        import math
        ratio = self.ratio
        if x.shape[1] == 4:
            gains = (0.34, 0.32, 0.30, 0.22)
        elif x.shape[1] == 1:
            gains = (0.5,)  # Generic anti-alias setting; NOT measured QB PAN MTF.
        else:
            raise ValueError("Expected four band-aligned residual channels or PAN")
        if any(n % ratio for n in x.shape[-2:]):
            raise ValueError("Integer-ratio sampling requires divisible dimensions")
        # Gaussian frequency response at target Nyquist: exp(-sigma^2*(pi/r)^2/2).
        sigmas = [math.sqrt(-2*math.log(g))*ratio/math.pi for g in gains]
        radius = math.ceil(3*max(sigmas))
        with torch.autocast(x.device.type, enabled=False):
            coordinate = torch.arange(-radius,radius+1,device=x.device,dtype=torch.float32)
            kernels = torch.stack([torch.exp(-coordinate.square()/(2*v*v)) for v in sigmas])
            kernels = kernels/kernels.sum(1,keepdim=True)
            h,w = x.shape[-2:]
            rows = torch.arange(-radius,h+radius,device=x.device).clamp(0,h-1)
            cols = torch.arange(-radius,w+radius,device=x.device).clamp(0,w-1)
            y = F.conv2d(x.float().index_select(-1,cols),kernels[:,None,None,:],groups=x.shape[1])
            y = F.conv2d(y.index_select(-2,rows),kernels[:,None,:,None],groups=x.shape[1])
            # Same source coordinate j*(N-1)/(M-1) as original align_corners=True.
            y = F.interpolate(y,scale_factor=1/ratio,mode="bilinear",align_corners=True)
        return y.to(x.dtype)


class SharedThreeStage(nn.Module):
    def __init__(self, channels, k, feedback):
        super().__init__()
        # Instantiate the common core first for identical B1/B1_AA initialization.
        self.core = RestoredPansharpeningNet(channels, k)
        self.error_encoder = nn.Conv2d(channels, channels, 3, padding=1)
        self.alpha = nn.Parameter(torch.tensor(0.1))
        self.observation = FFTObservation("QB").float()
        self.feedback = feedback
        self.margin = 5
        self._masks = {}
        self.register_buffer("phase_candidates", torch.tensor([[2,2],[1,2],[2,1]], dtype=torch.long))

    def samples(self, blurred):
        # Static phase slices keep all selection and indexing on the GPU.
        return torch.stack([blurred[...,2::4,2::4], blurred[...,1::4,2::4],
                            blurred[...,2::4,1::4]], dim=1)

    @torch.no_grad()
    def select_phase(self, lms, ms):
        # Inference geometry uses observed LMS/MS only; never test GT.
        with torch.autocast("cuda", enabled=False):
            views = self.samples(self.observation.blur(lms.float()))
        margin = self.margin if min(ms.shape[-2:]) > 2*self.margin else 0
        delta = views - ms.float()[:,None]
        if margin: delta = delta[...,margin:-margin,margin:-margin]
        return delta.square().mean((2,3,4)).argmin(1)

    def forward(self, pan, lms, ms, phase=None):
        current = lms
        if self.feedback and phase is None: phase = self.select_phase(lms, ms)
        for _ in range(3):
            if self.feedback:
                # Fixed FP32 sensor model, with gradients through current reconstruction.
                with torch.autocast("cuda", enabled=False):
                    views = self.samples(self.observation.blur(current.float()))
                    observed = views[torch.arange(len(ms),device=ms.device),phase]
                    error = ms.float() - observed
                    m = self.margin
                    geometry = (ms.shape[-2], ms.shape[-1], ms.device)
                    if geometry not in self._masks:
                        rows = torch.arange(ms.shape[-2],device=ms.device)
                        cols = torch.arange(ms.shape[-1],device=ms.device)
                        self._masks[geometry] = ((rows>=m)&(rows<ms.shape[-2]-m))[:,None] & ((cols>=m)&(cols<ms.shape[-1]-m))[None,:]
                    mask = self._masks[geometry]
                    error = F.interpolate(error*mask, size=current.shape[-2:],mode="bilinear",align_corners=False)
            else:
                error = torch.zeros_like(current)
            conditioned = current + self.error_encoder(error)
            proposal = self.core(pan, conditioned, ms)
            current = current + self.alpha * (proposal - current)
        return current

def build_model(variant, channels, k):
    if variant not in ("B1","B1_AA"): raise ValueError(variant)
    model = SharedThreeStage(channels,k,False)
    if variant == "B1_AA":
        for name, ratio in (("downsample1",4),("downsample2",4),
                            ("downsample100",2),("downsample200",2)):
            original = getattr(model.core,name)
            if original.scale_factor != 1/ratio or original.align_corners is not True:
                raise RuntimeError("Pinned core geometry changed")
            setattr(model.core, name, BandLimitedDownsample(ratio))
    return model

def check_downsampling():
    for ratio in (2,4):
        for channels in (1,4):
            op = BandLimitedDownsample(ratio).cuda()
            constant = torch.ones(1,channels,64,64,device="cuda",requires_grad=True)
            out = op(constant)
            torch.testing.assert_close(out,torch.ones_like(out),atol=2e-6,rtol=0)
            out.sum().backward()
            if not torch.isfinite(constant.grad).all(): raise RuntimeError("FIR backward")
            ramp = torch.arange(64,device="cuda").float()[None,None,None,:].expand(1,channels,64,64)
            expected = F.interpolate(ramp,scale_factor=1/ratio,mode="bilinear",align_corners=True)
            torch.testing.assert_close(op(ramp)[...,4:-4,4:-4],expected[...,4:-4,4:-4],atol=1e-5,rtol=1e-5)
            axis = torch.arange(64,device="cuda")
            checker = ((axis[:,None]+axis[None,:])%2).float()*2-1
            filtered = op(checker[None,None].expand(1,channels,64,64))
            if filtered[...,4:-4,4:-4].square().mean() >= .01:
                raise RuntimeError("Insufficient checkerboard attenuation")


def train_config(variant, seed):
    return {"variant":variant,"seed":seed,"stages":3,"shared_weights":True,
            "loss":"final_output_MSE","initial_alpha":0.1,
            "train_phase":"unused in both zero-feedback arms",
            "inference_phase":"unused by forward; input-only diagnostic during evaluation"}
'''

custom_hash = hashlib.sha256(PASTED_MODEL_SOURCE.encode()).hexdigest()
custom_path.parent.mkdir(parents=True, exist_ok=True)
if custom_path.exists() and sha(custom_path) != custom_hash:
    if RESUME_INPUT:
        raise ValueError(
            "RESUME_INPUT contains different model code. To avoid loading incompatible checkpoints, "
            "clear RESUME_INPUT and run this version as a separate experiment."
        )

    # Never overwrite old code/checkpoints. Reuse a prior code-change output only
    # when its saved model hash AND suite settings match this exact experiment;
    # otherwise select a new sibling output directory.
    base_out = OUT
    selected_out = None
    for index in range(1, 1000):
        candidate = base_out.with_name(f"{base_out.name}_codechange_{index:03d}")
        candidate_model = candidate / "code" / "B1_AA_models.py"
        candidate_config = candidate / "suite_config.json"
        if candidate_model.is_file() and candidate_config.is_file():
            try:
                same_settings = json.loads(candidate_config.read_text(encoding="utf-8")) == suite_settings
            except (OSError, ValueError):
                same_settings = False
            if same_settings and sha(candidate_model) == custom_hash:
                selected_out = candidate
                break
        if not candidate.exists():
            selected_out = candidate
            break
    if selected_out is None:
        raise RuntimeError("Could not find a safe output directory for the updated model code.")

    print(
        f"[OUTPUT] Existing model code differs; preserving {base_out} and using {selected_out}. "
        "Old checkpoints will not be resumed across this code change.",
        flush=True,
    )
    OUT = selected_out
    OUTPUT_DIR = str(OUT)
    OUT.mkdir(parents=True, exist_ok=True)
    write_json(OUT / "suite_config.json", suite_settings)
    SOURCE = OUT / "code" / "CtrS"
    custom_path = OUT / "code" / "B1_AA_models.py"
    custom_path.parent.mkdir(parents=True, exist_ok=True)
custom_path.write_text(PASTED_MODEL_SOURCE, encoding="utf-8")
files = ["src/ssamrn/__init__.py", "src/ssamrn/models/__init__.py",
         "src/ssamrn/data/__init__.py", "src/ssamrn/data/pancollection.py",
         "src/ssamrn/models/ssa_mrn.py", "src/ssamrn/models/pan_variants.py",
         "src/ssamrn/models/interp23.py", "src/ssamrn/observation.py",
         "src/ssamrn/metrics.py", "scripts/verify_observation.py"]
source_urls = {f"SSA-MRN/{name}":
               f"https://raw.githubusercontent.com/BIYONGHIYON/CtrS/{REPO_COMMIT}/SSA-MRN/{name}"
               for name in files}
source_urls["SSA-MRN/references/upstream/network.py"] = (
    f"https://raw.githubusercontent.com/zhouchuanxu/SSA-MRN/{UPSTREAM_COMMIT}/network.py")
source_state = OUT / "code" / "source.json"
old_source = json.loads(source_state.read_text()) if source_state.exists() else None
if old_source and (old_source["repo_commit"] != REPO_COMMIT or
                   old_source["upstream_commit"] != UPSTREAM_COMMIT):
    raise ValueError("Source revision changed; choose a new OUTPUT_DIR.")
for relative, url in source_urls.items():
    target = SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if old_source and sha(target) != old_source["files"][relative]["sha256"]:
            raise ValueError(f"Source was modified: {target}")
    else:
        temporary = target.with_suffix(".download")
        print(f"[DOWNLOAD] {relative}", flush=True)
        # Small pinned Python files only; a stalled connection fails within 30 seconds.
        with urllib.request.urlopen(url, timeout=30) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        print(f"[DOWNLOAD DONE] {relative}", flush=True)
        os.replace(temporary, target)
write_json(source_state, {"repo_commit": REPO_COMMIT, "upstream_commit": UPSTREAM_COMMIT,
                         "files": {name: {"url": url, "sha256": sha(SOURCE / name)}
                                   for name, url in source_urls.items()}})

data_info = {}
for split, path in PATHS.items():
    print(f"Checking {split}: {path}", flush=True)
    with h5py.File(path, "r") as h5:
        keys = ("pan", "lms", "ms", "gt") if split != "FR" else ("pan", "lms", "ms")
        shapes = {key: list(h5[key].shape) for key in keys}
        n, c, h, w = shapes["lms"]
        if (n < (5 if split == "RR" else 1) or c != (8 if SENSOR == "WV3" else 4)
                or h % 4 or w % 4 or shapes["pan"] != [n, 1, h, w]
                or shapes["ms"] != [n, c, h // 4, w // 4]
                or (split != "FR" and shapes["gt"] != shapes["lms"])):
            raise ValueError(f"Invalid NCHW arrays in {path}: {shapes}")
        if split in ("RR", "FR") and (h % 32 or w % 32):
            raise ValueError("Repository metrics require 32-pixel aligned full scenes.")
    data_info[split] = {"path": str(path), "bytes": path.stat().st_size,
                        "sha256": sha(path), "shapes": shapes}
if len({info["sha256"] for info in data_info.values()}) != 4:
    raise ValueError("Identical files found in different splits.")
write_json(OUT / "data_manifest.json", data_info)
print(json.dumps(data_info, indent=2), flush=True)
# Neither arm uses observation feedback or TRAIN GT phase calibration.
audit_path = OUT / "observation_train.json"
write_json(audit_path, {"status":"not_used", "phases":[],
    "reason":"Both arms preserve existing zero-feedback B1 forward"})
write_json(OUT / "plan_deviations.json", {
    "baseline":"existing shared-three-stage B1, zero error plus encoder bias",
    "candidate":"same B1; only four core downsample modules replaced",
    "k":K, "upsampling":"unchanged original bilinear",
    "QB_band_gains":[.34,.32,.30,.22], "PAN_gain":.5,
    "physical_status":"Gaussian feature-space proxy hypothesis; PAN gain generic, not calibrated",
    "coordinate":"j*(N-1)/(M-1); align_corners=True; replicate boundary; separable normalized positive Gaussian",
    "learned_kernel":"not enabled; deferred until fixed-filter comparison"})

selection = {"seed": 42, "protocol": "RR", "tiles": "full scene",
             "zero_based_scene_indices": sorted(np.random.default_rng(42).choice(
                 data_info["RR"]["shapes"]["pan"][0], 5, replace=False).tolist()),
             "order": "ascending scene index", "rgb_assumed_bands_zero_based": [2, 1, 0],
             "rule": "fixed before training and test scoring"}
selection_path = OUT / "selection.json"
if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
    raise ValueError("Existing example selection differs; choose a new OUTPUT_DIR.")
write_json(selection_path, selection)

# No normalized on-disk cache: workers stream original H5 directly into their GPU.
CACHE = OUT / "_unused_cache"  # Retained only for compatibility with config schema.
print("[START WORKERS] Direct H5 -> GPU FP32; no normalized disk copy",flush=True)

# Separate CUDA processes keep seeds and optimizer states independent on T4 x2.
WORKER = r'''
import gc, hashlib, json, math, os, random, shutil, sys, time, traceback
from pathlib import Path
import h5py
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib
matplotlib.use("Agg")
from PIL import Image, ImageDraw

C = json.loads(Path(sys.argv[1]).read_text())
sys.path.insert(0, str(Path(C["source"]) / "SSA-MRN/src"))
from ssamrn.models.ssa_mrn import RestoredPansharpeningNet
from ssamrn.models.pan_variants import HighFrequency
from ssamrn.data.pancollection import PanCollectionH5
from ssamrn.metrics import rr_metrics, fr_metrics

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(8 * 1024**2), b""): h.update(b)
    return h.hexdigest()

def write(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)

def cpu(value):
    if isinstance(value, torch.Tensor): return value.detach().cpu().clone()
    if isinstance(value, dict): return {k: cpu(v) for k, v in value.items()}
    if isinstance(value, list): return [cpu(v) for v in value]
    if isinstance(value, tuple): return tuple(cpu(v) for v in value)
    return value

def save(value, path):
    tmp = path.with_name(path.name + ".tmp")
    torch.save(value, tmp); os.replace(tmp, path)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def model_for(config):
    return PASTED.build_model(config["variant"], config["channels"], config["k"])

def load_gpu_cache():
    size = sum(np.prod(v)*4 for split in ("train","val") for v in C["data"][split]["shapes"].values())
    free = torch.cuda.mem_get_info()[0]
    if size > free*.65: raise RuntimeError("Full FP32 dataset exceeds GPU activation reserve")
    result = {}; total = sum(C["data"][split]["shapes"][key][0] for split in ("train","val") for key in KEYS)
    done = 0; last = 0
    try:
        for split in ("train","val"):
            result[split] = {}
            with h5py.File(C["data"][split]["path"],"r") as h5:
                for key in KEYS:
                    shape = C["data"][split]["shapes"][key]
                    tensor = torch.empty(shape,device="cuda",dtype=torch.float32)
                    result[split][key] = tensor
                    for start in range(0,shape[0],128):
                        chunk = np.asarray(h5[key][start:start+128],dtype=np.float32)
                        if not np.isfinite(chunk).all(): raise ValueError(f"Nonfinite {split}/{key}")
                        tensor[start:start+len(chunk)].copy_(torch.from_numpy(chunk))
                        done += len(chunk)
                        if time.monotonic()-last >= 2 or start+len(chunk)==shape[0]:
                            status("loading_cache",cache_array=f"{split}/{key}",cache_percent=100*done/total)
                            print(f"GPU load {split}/{key}: {start+len(chunk)}/{shape[0]} | total {100*done/total:.1f}%",flush=True)
                            last=time.monotonic()
                    tensor.div_(C["scale"])
        torch.cuda.synchronize()
        return result
    except torch.cuda.OutOfMemoryError:
        result.clear()
        if "tensor" in locals(): del tensor
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("GPU dataset allocation failed; CPU fallback disabled")

def batches(split, batch_size, generator=None, shuffle=False, order=None, start_batch=0):
    n = C["data"][split]["shapes"]["gt"][0]
    if GPU_DATA is not None:
        if order is None:
            order = torch.randperm(n, generator=generator, device="cuda") if shuffle else torch.arange(n, device="cuda")
        for start in range(start_batch * batch_size, n, batch_size):
            indices = order[start:start + batch_size]
            batch = {k: v.index_select(0, indices) for k, v in GPU_DATA[split].items()}
            if split == "train": batch["phase"] = TRAIN_PHASE.index_select(0, indices)
            yield batch
    else:
        raise RuntimeError("GPU residency is required")

def layout(b):
    return {k: (v.contiguous(memory_format=FORMAT) if v.ndim == 4 else v) for k, v in b.items()}

def build():
    model = model_for(C).cuda().to(memory_format=FORMAT)
    model.observation.fft_enabled = C["fft_observation"]
    return model


def make_optimizer(model):
    if C["fused_adam"]:
        return torch.optim.Adam(model.parameters(), lr=C["lr"], fused=True)
    return torch.optim.Adam(model.parameters(), lr=C["lr"], foreach=True)


def epoch_lr(epoch_index):
    if epoch_index < C["hold_lr_epochs"]:
        return C["lr"]
    progress = (epoch_index - C["hold_lr_epochs"] + 1) / (C["epochs"] - C["hold_lr_epochs"])
    return C["min_lr"] + .5 * (C["lr"] - C["min_lr"]) * (1 + math.cos(math.pi * progress))

def tune_micro():
    probe = {k: GPU_DATA["train"][k][:C["batch_size"]] for k in KEYS}
    probe = layout(probe); n = len(probe["gt"])
    records = []
    candidates = ([min(n, C["micro_batch"])] if not C["auto_micro"] else
                  sorted(set(min(n, b) for b in [C["batch_size"], 16, 8, 4, 1]), reverse=True))
    for micro in candidates:
        model = optimizer = scaler = pred = loss = None
        try:
            seed_all(C["seed"]); model = build()
            optimizer = make_optimizer(model)
            scaler = torch.amp.GradScaler("cuda", enabled=C["amp"])
            for repeat in range(3):
                torch.cuda.synchronize(); started = time.monotonic()
                optimizer.zero_grad(set_to_none=True)
                for offset in range(0, n, micro):
                    end = min(offset + micro, n)
                    with torch.autocast("cuda", dtype=torch.float16, enabled=C["amp"]):
                        pred = model(*(probe[k][offset:end] for k in INPUTS))
                        loss = F.mse_loss(pred.float(), probe["gt"][offset:end]) * ((end - offset) / n)
                    scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                torch.cuda.synchronize()
                if repeat == 2:
                    if not torch.isfinite(loss).item(): raise RuntimeError("Nonfinite benchmark loss")
                    records.append({"micro_batch": micro, "seconds_per_effective_batch": time.monotonic() - started})
            # A full effective batch avoids extra launches; only test smaller
            # successful sizes near it. OOM candidates always continue downward.
            if len(records) >= 2: break
        except torch.cuda.OutOfMemoryError:
            pass
        finally:
            del model, optimizer, scaler, pred, loss
            gc.collect(); torch.cuda.empty_cache()
    del probe
    if not records: raise RuntimeError("Configured micro batch does not fit. Set LEGACY_MICRO_BATCH_SIZE=16 for BOTH arms and choose a new OUTPUT_DIR. GPU residency stays enabled.")
    micro = min(records, key=lambda x: x["seconds_per_effective_batch"])["micro_batch"]
    write(RUN / "benchmark.json", {"candidates": records, "chosen_micro_batch": micro,
          "cache": "GPU" if GPU_DATA is not None else "mmap",
          "memory_format": str(FORMAT)})
    return micro

def status(phase, **values):
    state = {"run_id": RUN.name, "phase": phase,
          "device": torch.cuda.get_device_name(0), "epoch_target": C["epochs"],
          "allocated_gb": torch.cuda.memory_allocated() / 1024**3,
          "reserved_gb": torch.cuda.memory_reserved() / 1024**3,
          "peak_gb": torch.cuda.max_memory_allocated() / 1024**3,
          "updated_unix": time.time(), **values}
    write(RUN / "status.json",state)
    print("[STATUS] "+json.dumps(state,allow_nan=False),flush=True)

@torch.inference_mode()
def validate(model):
    model.eval(); sse = torch.zeros((), device="cuda"); count = 0
    psnr = torch.zeros((), device="cuda"); sam = torch.zeros((), device="cuda")
    samples = 0
    for b in batches("val", C["eval_batch"]):
        b = layout(b)
        # FP32 validation makes best selection independent of AMP inference.
        pred = model(*(b[k] for k in INPUTS)).float(); gt = b["gt"]
        diff = (pred - gt).square(); sse += diff.sum(); count += gt.numel()
        band_mse = diff.mean((2, 3)).clamp_min(1e-30)
        psnr += (-10 * band_mse.log10()).mean(1).sum(); samples += len(gt)
        norms = pred.norm(dim=1) * gt.norm(dim=1); valid = norms > 0
        angles = torch.rad2deg(torch.acos(((pred * gt).sum(1) / norms.clamp_min(1e-30)).clamp(-1, 1)))
        valid_counts = valid.flatten(1).sum(1)
        # Zero-valid pixels propagate NaN to the epoch-level finite check.
        sam += ((angles * valid).flatten(1).sum(1) / valid_counts).sum()
    result = {"val_mse": (sse / count).item(), "val_psnr_peak1": (psnr / samples).item(),
              "val_sam_deg": (sam / samples).item()}
    if not all(math.isfinite(v) for v in result.values()):
        raise RuntimeError("Invalid validation metrics")
    return result

def measure_test_inference(model):
    timings = {}
    model.eval()
    with torch.inference_mode():
        for protocol in ("RR","FR"):
            with h5py.File(C["data"][protocol]["path"],"r") as h5:
                inputs = [torch.from_numpy(np.asarray(h5[k][:1],dtype=np.float32)/C["scale"]).cuda().contiguous(memory_format=FORMAT) for k in INPUTS]
            model(*inputs)  # warmup; no scoring or example selection
            torch.cuda.synchronize(); begin = time.monotonic()
            for _ in range(3): model(*inputs)
            torch.cuda.synchronize()
            timings[protocol] = (time.monotonic()-begin)/3
    write(RUN / "test_timing_probe.json",{"seconds_per_scene":timings,
        "scope":"random initialization, inference only; metrics/I/O/packaging unmeasured", "scene_index":0})
    return sum(timings[p]*C["data"][p]["shapes"]["pan"][0] for p in ("RR","FR"))


def train():
    status("benchmark")
    micro = tune_micro()
    seed_all(C["seed"])
    gen = torch.Generator(device="cuda").manual_seed(C["seed"])
    model = build()
    test_inference_estimate = measure_test_inference(model)
    opt = make_optimizer(model)
    core_hash = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        core_hash.update(name.encode())
        core_hash.update(value.detach().cpu().contiguous().numpy().tobytes())
    write(RUN / "initialization.json", {"seed": C["seed"], "core_sha256": core_hash.hexdigest(),
          "excluded_candidate_layers": []})
    scaler = torch.amp.GradScaler("cuda", enabled=C["amp"])
    start, best, best_epoch, history, resume_mid = 0, float("inf"), 0, [], None
    if (RUN / "latest.pt").exists():
        ck = torch.load(RUN / "latest.pt", map_location="cpu", weights_only=True)
        if ck["config"] != C:
            raise ValueError("Resume configuration differs; use a new output directory")
        model.load_state_dict(ck["model"], strict=True)
        opt.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["cpu_rng"])
        torch.cuda.set_rng_state_all(ck["cuda_rng"])
        random.setstate(ck["python_rng"])
        np_state = ck["numpy_rng"]
        np.random.set_state((np_state[0],np.array(np_state[1],dtype=np.uint32),np_state[2],np_state[3],np_state[4]))
        gen.set_state(ck["loader_rng"])
        start, best, best_epoch, history = ck["epoch"], ck["best_mse"], ck["best_epoch"], ck["history"]
        resume_mid = ck.get("mid_epoch")
        ck_best_recovery = ck.get("best_recovery")
        if best_epoch and ck_best_recovery is None:
            raise RuntimeError("Missing original best recovery in latest")
        del ck
        if best_epoch:
            save(ck_best_recovery, RUN / "best.pt")
    total_batches = math.ceil(C["data"]["train"]["shapes"]["gt"][0] / C["batch_size"])

    def snapshot(completed_epoch, mid=None):
        return {"config": C, "epoch": completed_epoch, "best_mse": best, "best_epoch": best_epoch,
            "model": cpu(model.state_dict()), "optimizer": cpu(opt.state_dict()),
            "scaler": scaler.state_dict(), "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(), "loader_rng": gen.get_state(),
            "python_rng": random.getstate(),
            "numpy_rng": (np.random.get_state()[0],np.random.get_state()[1].tolist(),
                np.random.get_state()[2],np.random.get_state()[3],np.random.get_state()[4]),
            "history": history, "mid_epoch": mid}

    if not (RUN / "latest.pt").exists():
        initial = snapshot(0)
        initial["best_recovery"] = None
        save(initial,RUN / "latest.pt")
    for epoch in range(start, C["epochs"]):
        lr = epoch_lr(epoch)
        for group in opt.param_groups:
            group["lr"] = lr
        n_samples = C["data"]["train"]["shapes"]["gt"][0]
        if resume_mid is not None:
            if resume_mid["epoch_index"] != epoch:
                raise ValueError("Invalid mid-epoch recovery point")
            order = resume_mid["order"].cuda()
            completed_batches = resume_mid["batches_done"]
            if not (0 <= completed_batches <= total_batches) or len(order) != n_samples:
                raise ValueError("Invalid recovered batch cursor")
            sse = resume_mid["sse"].cuda()
            count = resume_mid["count"]
            seconds_before = resume_mid["train_seconds"]
            order_hash = resume_mid["sample_order_sha256"]
            resume_mid = None
        else:
            order = torch.randperm(n_samples, generator=gen, device="cuda")
            order_hash = hashlib.sha256(order.cpu().numpy().tobytes()).hexdigest()
            completed_batches = 0
            sse = torch.zeros((), device="cuda")
            count = 0
            seconds_before = 0.
        model.train()
        torch.cuda.synchronize()
        tick = time.monotonic()
        last_status = last_checkpoint = tick
        last_step = completed_batches
        status("train", epoch=epoch + 1, batch=completed_batches, batches=total_batches, lr=lr)
        for step, b in enumerate(batches("train", C["batch_size"], order=order,
                                       start_batch=completed_batches), completed_batches + 1):
            b = layout(b)
            n = len(b["gt"])
            opt.zero_grad(set_to_none=True)
            for offset in range(0, n, micro):
                end = min(offset + micro, n)
                with torch.autocast("cuda", dtype=torch.float16, enabled=C["amp"]):
                    pred = model(*(b[k][offset:end] for k in INPUTS), phase=b["phase"][offset:end])
                    loss = F.mse_loss(pred.float(), b["gt"][offset:end])
                scaler.scale(loss * ((end - offset) / n)).backward()
                # Reuse the detached MSE; no second square/sum or batch .item().
                sse += loss.detach() * b["gt"][offset:end].numel()
                count += b["gt"][offset:end].numel()
            scaler.step(opt)
            scaler.update()
            now = time.monotonic()
            last_step = step
            if now - last_status >= C["status_seconds"]:
                # Periodic boundary makes ETA describe completed GPU work, not queued launches.
                torch.cuda.synchronize()
                elapsed = seconds_before + time.monotonic() - tick
                prior = [r["train_seconds"] for r in history[-5:]]
                measured = elapsed / step * total_batches
                estimate = float(np.mean(prior)) if prior else measured
                train_left = max(0.,estimate-elapsed)+(C["epochs"]-epoch-1)*estimate
                extra = (C["epochs"]-epoch)*float(np.mean([r["validation_seconds"]+r.get("checkpoint_seconds",0) for r in history[-5:]])) if history else 0.
                status("train", epoch=epoch + 1, batch=step, batches=total_batches, lr=lr,
                    train_remaining_seconds=train_left,
                    training_finish_unix=time.time()+train_left+extra if history else None,
                    exit_finish_unix=time.time()+train_left+extra+test_inference_estimate,
                    exit_eta_scope="provisional GPU finish: measured training and test inference; validation/checkpoint included after first epoch; test metrics/I/O/packaging unmeasured",
                    eta_scope="measured training; validation/checkpoint added after first epoch")
                last_status = time.monotonic()
            if now - last_checkpoint >= C["checkpoint_seconds"] and step < total_batches:
                torch.cuda.synchronize()
                elapsed = seconds_before + time.monotonic() - tick
                status("checkpoint", epoch=epoch + 1, batch=step, batches=total_batches, lr=lr)
                mid = {"epoch_index": epoch, "batches_done": step, "sse": sse.detach().cpu(),
                    "count": count, "order": order.detach().cpu(), "train_seconds": elapsed,
                    "sample_order_sha256": order_hash}
                recovery = snapshot(epoch, mid)
                recovery["best_recovery"] = torch.load(RUN / "best.pt",map_location="cpu",weights_only=True) if best_epoch else None
                save(recovery, RUN / "latest.pt")
                del recovery
                last_checkpoint = time.monotonic()
                status("train", epoch=epoch + 1, batch=step, batches=total_batches, lr=lr)
        train_mse = (sse / count).item()
        if not math.isfinite(train_mse):
            raise RuntimeError("Nonfinite training loss")
        torch.cuda.synchronize()
        training_seconds = seconds_before + time.monotonic() - tick
        status("validation", epoch=epoch + 1, batch=last_step, batches=total_batches, lr=lr)
        val_started = time.monotonic()
        validation = validate(model)
        validation_seconds = time.monotonic() - val_started
        improved = validation["val_mse"] < best
        if improved:
            best, best_epoch = validation["val_mse"], epoch + 1
        row = {"epoch": epoch + 1, "train_mse": train_mse, **validation, "lr": lr,
            "seconds": training_seconds + validation_seconds, "train_seconds": training_seconds,
            "validation_seconds": validation_seconds,
            "train_samples_per_second": n_samples / max(training_seconds, 1e-9),
            "scaler_scale": scaler.get_scale(), "micro_batch": micro,
            "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024**3,
            "sample_order_sha256": order_hash}
        history.append(row)
        status("checkpoint", epoch=epoch + 1, batch=total_batches, batches=total_batches, metrics=row, lr=lr)
        checkpoint_started = time.monotonic()
        ck = snapshot(epoch + 1)
        if improved:
            save(ck, RUN / "best.pt")
        ck["best_recovery"] = torch.load(RUN / "best.pt",map_location="cpu",weights_only=True)
        save(ck, RUN / "latest.pt")
        if not improved:
            # Refresh atomically every epoch while retaining the actual best weights/epoch.
            shutil.copy2(RUN / "best.pt", RUN / "best.pt.tmp")
            os.replace(RUN / "best.pt.tmp", RUN / "best.pt")
        checkpoint_seconds = time.monotonic() - checkpoint_started
        row["checkpoint_seconds"] = checkpoint_seconds
        if epoch + 1 == C["hold_lr_epochs"]:
            # Preserve the actual validation-best of the first 100 epochs for later analysis.
            shutil.copy2(RUN / "best.pt", RUN / "best_first100.pt")
        write(RUN / "history.json", history)
        remaining_epochs = C["epochs"]-epoch-1
        epoch_mean = float(np.mean([r["seconds"]+r.get("checkpoint_seconds",0) for r in history[-5:]]))
        status("epoch_complete",epoch=epoch+1,batch=total_batches,batches=total_batches,
            metrics=row,training_finish_unix=time.time()+remaining_epochs*epoch_mean,
            exit_finish_unix=time.time()+remaining_epochs*epoch_mean+test_inference_estimate,
            exit_eta_scope="recent measured epochs plus RR/FR inference probe; test metrics/I/O/packaging unmeasured")
        del ck, order
        print(json.dumps(row), flush=True)
    write(RUN / "training_complete.json", {"config": C, "epoch": C["epochs"],
        "best_epoch": best_epoch, "best_mse": best})
    del model, opt, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return history


def panels(sample, prediction, path, index):
    bands = [2, 1, 0]; ref = sample["gt"][bands].transpose(1, 2, 0)
    lo, hi = np.percentile(ref, [1, 99], axis=(0, 1))
    def rgb(a):
        a = a[bands].transpose(1, 2, 0)
        return Image.fromarray((np.clip((a - lo) / np.maximum(hi - lo, 1e-8), 0, 1) * 255).astype("uint8"))
    pan = sample["pan"][0]; p0, p1 = np.percentile(pan, [1, 99])
    gray = Image.fromarray((np.clip((pan - p0) / max(p1 - p0, 1e-8), 0, 1) * 255).astype("uint8")).convert("RGB")
    h, w = pan.shape; canvas = Image.new("RGB", (4 * w, h + 40), "white")
    draw = ImageDraw.Draw(canvas)
    for j, (name, image) in enumerate([("LR MS", rgb(sample["ms"])), ("PAN", gray),
                                      ("Prediction", rgb(prediction)), ("GT", rgb(sample["gt"]))]):
        canvas.paste(image.resize((w, h), Image.Resampling.NEAREST), (j * w, 40))
        draw.text((j * w + 6, 5), name, fill="black")
    draw.text((6, 24), f'{C["sensor"]} RR scene {index} | shared GT 1-99% stretch | full scene', fill="black")
    canvas.save(path)
    return {"lo": lo.tolist(), "hi": hi.tolist(), "pan_lo_hi": [float(p0), float(p1)]}

def evaluate():
    ck = torch.load(RUN / "best.pt", map_location="cpu", weights_only=True)
    if ck["config"] != C: raise ValueError("Best configuration mismatch")
    model = build(); model.load_state_dict(ck["model"], strict=True); model.eval()
    result = {"config": C, "best_epoch": ck["epoch"], "best_mse": ck["best_mse"],
              "checkpoints": {kind: {"sha256": sha(RUN / f"{kind}.pt"),
                  "bytes": (RUN / f"{kind}.pt").stat().st_size,
                  "epoch": torch.load(RUN / f"{kind}.pt",map_location="cpu",weights_only=True)["epoch"],
                  "original_path":str(RUN / f"{kind}.pt")} for kind in ("best", "latest")},
              "metrics": {}, "images": [], "test_precision": "FP32", "full_scene": True}
    del ck
    for protocol in ("RR", "FR"):
        rows = []; tick = time.monotonic()
        if C["save_predictions"]: (RUN / "predictions" / protocol).mkdir(parents=True, exist_ok=True)
        with h5py.File(C["data"][protocol]["path"], "r") as h5:
            keys = INPUTS + (("gt",) if protocol == "RR" else ())
            for index in range(len(h5["pan"])):
                measured = (time.monotonic()-tick)/index if index else None
                eta = {"exit_finish_unix":time.time()+measured*(len(h5["pan"])-index),
                       "exit_eta_scope":"FR remaining scenes; packaging excluded"} if measured and protocol=="FR" else {}
                status("test_" + protocol, scene=index + 1, scenes=len(h5["pan"]), **eta)
                arrays = {k: np.asarray(h5[k][index], dtype=np.float32) for k in keys}
                if any(not np.isfinite(v).all() for v in arrays.values()): raise ValueError("Nonfinite test input")
                inputs = [torch.from_numpy(arrays[k] / C["scale"]).unsqueeze(0).cuda().contiguous(
                          memory_format=FORMAT) for k in INPUTS]
                begin = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True)
                with torch.inference_mode():
                    begin.record()
                    prediction = model(*inputs).float()
                    end.record(); end.synchronize()
                    inference_ms = begin.elapsed_time(end)
                    phase = model.select_phase(inputs[1], inputs[2])
                    observed = model.samples(model.observation.blur(prediction))[
                        torch.arange(len(prediction),device="cuda"),phase]
                    m = model.margin
                    input_mse = F.mse_loss(observed[...,m:-m,m:-m],inputs[2][...,m:-m,m:-m]).item()
                    pred = prediction[0].cpu().numpy() * C["scale"]
                del prediction, observed
                if not np.isfinite(pred).all(): raise RuntimeError("Nonfinite prediction")
                if protocol == "RR":
                    metrics = rr_metrics(arrays["gt"], pred)
                    metrics["MSE_peak1"] = float(np.mean(((pred.astype(np.float64) - arrays["gt"]) / C["scale"])**2))
                    metrics["PSNR_sensor_peak"] = metrics["PSNR"] + 20 * math.log10(C["scale"] / 2047.)
                    if index in C["selection"]["zero_based_scene_indices"]:
                        name = f"RR_scene_{index:02}.png"
                        result["images"].append({"scene_index": index, "file": name,
                                                "display": panels(arrays, pred, RUN / name, index)})
                else: metrics = fr_metrics(pred, arrays["lms"], arrays["ms"], arrays["pan"])
                metrics["MS_consistency_MSE_peak1_interior"] = input_mse
                metrics["inference_ms"] = inference_ms
                if not all(math.isfinite(v) for v in metrics.values()): raise RuntimeError("Nonfinite test metric")
                if C["save_predictions"]:
                    np.savez_compressed(RUN / "predictions" / protocol / f"scene_{index:03}.npz",
                                        prediction_dn=pred, scene_index=index, scale=C["scale"])
                rows.append({"scene_index": index, **metrics})
                write(RUN / f"{protocol}_per_scene.json", rows)
                del inputs
        result["metrics"][protocol] = {"count": len(rows), "rows": rows,
             "mean": {k: float(np.mean([r[k] for r in rows])) for k in rows[0] if k != "scene_index"},
             "seconds": time.monotonic() - tick}
    if len(result["images"]) != 5: raise RuntimeError("Incomplete fixed RR examples")
    write(RUN / "results.json", result)
    write(RUN / "evaluation_complete.json", {"best_sha256": result["checkpoints"]["best"]["sha256"],
          "RR": result["metrics"]["RR"]["count"], "FR": result["metrics"]["FR"]["count"]})
    status("complete", epoch=C["epochs"], best_epoch=result["best_epoch"], metrics=result["metrics"]["RR"]["mean"])

if __name__ == "__main__":
    C = json.loads(Path(sys.argv[1]).read_text()); RUN = Path(C["run_dir"])
    RUN.mkdir(parents=True, exist_ok=True)
    if C["mode"] in ("B1", "B1_AA"):
        import importlib.util
        spec = importlib.util.spec_from_file_location("B1_AA_models", C["B1_AA_source"])
        PASTED = importlib.util.module_from_spec(spec)
        # The original setup's constants are available to its function bodies.
        PASTED.__dict__.update({"SENSOR": C["sensor"], "K": C["k"], "EPOCHS": C["epochs"],
            "EFFECTIVE_BATCH_SIZE": C["batch_size"], "MICRO_BATCH_SIZE": C["micro_batch"],
            "LEARNING_RATE": C["lr"], "OUTPUT_DIR": str(RUN.parent),
            "TRAIN_H5": C["data"]["train"]["path"], "VAL_H5": C["data"]["val"]["path"],
            "SOURCE_ROOT": str(Path(C["source"]) / "SSA-MRN/src")})
        spec.loader.exec_module(PASTED)
        C["reference_plan_config"] = PASTED.train_config(C["variant"], C["seed"])
    KEYS = ("pan", "lms", "ms", "gt"); INPUTS = KEYS[:3]
    torch.cuda.set_device(0); torch.set_num_threads(2)
    torch.use_deterministic_algorithms(C["controlled"])
    torch.backends.cudnn.benchmark = not C["controlled"]
    torch.backends.cuda.matmul.allow_tf32 = not C["controlled"]
    torch.backends.cudnn.allow_tf32 = not C["controlled"]
    FORMAT = torch.channels_last if C["channels_last"] else torch.contiguous_format
    GPU_DATA = None; LOADERS = {}
    try:
        run_started = time.monotonic()
        status("smoke_checks")
        seed_all(C["seed"])
        first = PASTED.build_model("B1", C["channels"], C["k"])
        seed_all(C["seed"])
        second = PASTED.build_model("B1_AA", C["channels"], C["k"])
        if any(not torch.equal(v, second.state_dict()[k]) for k,v in first.state_dict().items()):
            raise RuntimeError("B1/B1_AA parameter initialization differs")
        del first
        second = second.cuda()
        if C["fft_observation"]:
            # Prove the optimization preserves the FIR including replication boundaries.
            obs = second.observation
            probe = torch.linspace(0, 1, 4*64*64, device="cuda").reshape(1,4,64,64).requires_grad_()
            weights = torch.linspace(.1, 1, probe.numel(), device="cuda").reshape_as(probe)
            obs.fft_enabled = False
            ref = obs.blur(probe)
            ref_grad = torch.autograd.grad((ref * weights).sum(), probe)[0]
            obs.fft_enabled = True
            got = obs.blur(probe)
            got_grad = torch.autograd.grad((got * weights).sum(), probe)[0]
            torch.testing.assert_close(got, ref, rtol=3e-5, atol=3e-6)
            torch.testing.assert_close(got_grad, ref_grad, rtol=8e-5, atol=1e-5)
            write(RUN / "fft_parity.json", {"forward_max_abs": (got-ref).abs().max().item(),
                "backward_max_abs": (got_grad-ref_grad).abs().max().item(),
                "tested_geometry": [64,64], "boundary": "replicate", "precision": "FP32",
                "scope": "numerical FIR equivalence; not a training performance result"})
            del probe, weights, ref, got, ref_grad, got_grad
        with h5py.File(C["data"]["train"]["path"], "r") as h5:
            example = {k: torch.from_numpy(np.asarray(h5[k][:1],dtype=np.float32)/C["scale"]).cuda() for k in KEYS}
        pred = second(*(example[k] for k in INPUTS))
        if pred.shape != example["gt"].shape: raise RuntimeError("Smoke output shape mismatch")
        F.mse_loss(pred, example["gt"]).backward()
        gradients = [p.grad for p in second.parameters() if p.grad is not None]
        if not gradients or not all(torch.isfinite(g).all().item() for g in gradients):
            raise RuntimeError("Smoke gradient check failed")
        # Zero input intentionally yields zero encoder-weight gradient in both B1 arms.
        PASTED.check_downsampling()
        write(RUN / "resampling_checks.json", {"constant":True,"ramp_phase":True,
            "checkerboard_attenuation":True,"finite_backward":True})
        write(RUN / "smoke_checks.json", {"matching_initialization":True,"shape":list(pred.shape),
              "finite_gradients":True,"feedback_gradient":False,"optimizer_steps":0})
        del second, example, pred, gradients
        gc.collect(); torch.cuda.empty_cache()
        status("loading_cache")
        GPU_DATA = load_gpu_cache()
        profile = json.loads(Path(C["observation_profile"]).read_text())
        if sha(C["observation_profile"]) != C["observation_profile_sha256"]:
            raise ValueError("Observation manifest changed")
        TRAIN_PHASE = torch.zeros(C["data"]["train"]["shapes"]["gt"][0],device="cuda",dtype=torch.long)
        train()
        training_wall_seconds = time.monotonic() - run_started
        for loader in LOADERS.values():
            if hasattr(loader.dataset, "close"): loader.dataset.close()
        GPU_DATA = None; del TRAIN_PHASE; LOADERS.clear(); gc.collect(); torch.cuda.empty_cache()
        evaluate()
        write(RUN / "session_timing.json", {"training_wall_seconds_this_session": training_wall_seconds,
              "total_wall_seconds_this_session": time.monotonic() - run_started,
              "scope": "cache load + benchmark + completed/replayed training + checkpoint saves + test; resumed session is not whole-experiment time"})
    except BaseException:
        write(RUN / "failure.json", {"traceback": traceback.format_exc(), "time": time.time()})
        traceback.print_exc(); sys.exit(1)
'''

worker_path = OUT / "code" / "kaggle_worker.py"
worker_hash = hashlib.sha256(WORKER.encode()).hexdigest()
if worker_path.exists() and sha(worker_path) != worker_hash:
    raise ValueError("Worker code changed; choose a new OUTPUT_DIR.")
worker_path.write_text(WORKER, encoding="utf-8")
gpu_count = torch.cuda.device_count() if USE_ALL_GPUS else 1
if len(TRAINING_MODES) == 2:
    gpu_count = min(gpu_count, 2)
    if gpu_count == 2:
        a, b = (torch.cuda.get_device_properties(i) for i in (0, 1))
        if a.name != b.name or a.total_memory != b.total_memory:
            raise ValueError("Controlled parallel comparison requires two equivalent GPUs.")
    print("Pair assignment: GPU 0=B1; GPU 1=B1_AA (swap optional)" if gpu_count == 2 else
          "Only one GPU: paired modes will run sequentially, not concurrently.")
configs = []
for mode, seed, variant in [(m, s, v) for m in TRAINING_MODES for s in SEEDS
                           for v in [m]]:
        run_id = f"{mode}_{SENSOR}_{variant}_k{K}_s{seed}"
        config = {"sensor": SENSOR, "k": K, "channels": 8 if SENSOR == "WV3" else 4,
                  "mode": mode, "controlled": CONTROLLED_COMPARISON,
                  "epochs": EPOCHS, "seed": seed, "variant": variant,
                  "hold_lr_epochs": HOLD_LR_EPOCHS, "min_lr": MIN_LEARNING_RATE,
                  "status_seconds": STATUS_INTERVAL_SECONDS, "checkpoint_seconds": CHECKPOINT_INTERVAL_SECONDS,
                  "fused_adam": FUSED_ADAM, "fft_observation": USE_FFT_OBSERVATION,
                  "batch_size": EFFECTIVE_BATCH_SIZE, "lr": LEARNING_RATE,
                  "micro_batch": min(LEGACY_MICRO_BATCH_SIZE, EFFECTIVE_BATCH_SIZE),
                  "auto_micro": not CONTROLLED_COMPARISON,
                  "eval_batch": min(LEGACY_MICRO_BATCH_SIZE, EFFECTIVE_BATCH_SIZE),
                  "amp": AMP,
                  "channels_last": CHANNELS_LAST,
                  "gpu_cache": GPU_CACHE, "workers": NUM_WORKERS, "scale": SCALE,
                  "save_predictions": SAVE_PREDICTIONS, "data": data_info,
                  "cache": str(CACHE), "source": str(SOURCE), "run_dir": str(OUT / "runs" / run_id),
                  "selection": selection, "repo_commit": REPO_COMMIT,
                  "plan_commit": PLAN_COMMIT,
                  "B1_AA_source": str(custom_path), "B1_AA_source_sha256": custom_hash,
                  "upstream_commit": UPSTREAM_COMMIT, "worker_sha256": worker_hash,
                  "observation_profile": str(audit_path), "observation_profile_sha256": sha(audit_path),
                  "model_definition": "3 shared zero-feedback B1 stages; B1_AA changes only core downsampling",
                  "numerics": ("Controlled: equal AMP/layout/micro batch, deterministic kernels, no TF32; not legacy bitwise parity" if CONTROLLED_COMPARISON
                               else "Speed profile: both arms use AMP/channels-last and measured micro batch; micro batches may differ")}
        cfg_path = OUT / "code" / f"{run_id}.json"
        if cfg_path.exists() and json.loads(cfg_path.read_text()) != config:
            raise ValueError(f"Settings changed for {run_id}; choose a new OUTPUT_DIR.")
        write_json(cfg_path, config)
        configs.append((config, cfg_path))
write_json(OUT / "environment.json", {"torch": str(torch.__version__), "cuda": torch.version.cuda,
           "gpus": [torch.cuda.get_device_name(i) for i in range(gpu_count)],
           "python": sys.version, "parallel_workers": min(gpu_count, len(configs)),
           "swap_gpus": SWAP_GPUS,
           "old_checkpoints": "legacy not imported; matching suite Output restores full optimizer/scaler/RNG/batch cursor"})

def _progress_label(state):
    """Return a phase-local percentage and a compact status label."""
    phase = state.get("phase", "starting")
    if phase == "complete":
        return 100.0, "완료"
    if phase == "loading_cache" and "cache_percent" in state:
        return float(state["cache_percent"]), f"GPU 데이터 로드 {state['cache_array']}"
    if phase == "train":
        epoch = int(state.get("epoch", 0))
        batches = max(1, int(state.get("batches", 1)))
        batch = int(state.get("batch", 0))
        pct = 100.0 * ((max(epoch - 1, 0) + min(max(batch, 0), batches) / batches) / max(EPOCHS, 1))
        return min(100.0, pct), f"학습 epoch {epoch}/{EPOCHS}, batch {batch}/{batches}"
    if phase in ("validation", "checkpoint", "epoch_complete"):
        epoch = int(state.get("epoch", 0))
        pct = 100.0 * epoch / max(EPOCHS, 1)
        return min(100.0, pct), f"{phase} epoch {epoch}/{EPOCHS}"
    if phase in ("test_RR", "test_FR") and "scene" in state:
        scenes = max(1, int(state.get("scenes", 1)))
        scene = int(state.get("scene", 0))
        pct = 100.0 * min(max(scene, 0), scenes) / scenes
        return pct, f"{phase[5:]} 평가 scene {scene}/{scenes}"
    labels = {
        "starting": "시작 준비", "smoke_checks": "모델 사전 검사",
        "benchmark": "배치 크기 측정", "loading_cache": "GPU 데이터 로드 준비",
        "checkpoint": "체크포인트 저장", "status_update": "상태 갱신 중",
    }
    return 0.0, labels.get(phase, phase)


pending = list(configs)
active = {}
finished = []
try:
    while pending or active:
        for device in range(gpu_count):
            if device in active or not pending:
                continue
            if gpu_count == 2 and len(TRAINING_MODES) == 2:
                reference_device = 1 if SWAP_GPUS else 0
                assigned_mode = "B1" if device == reference_device else "B1_AA"
                eligible = next((i for i, (c, p) in enumerate(pending) if c["mode"] == assigned_mode), None)
                if eligible is None: continue
            else: eligible = 0
            config, cfg_path = pending.pop(eligible)
            run = Path(config["run_dir"]); run.mkdir(parents=True, exist_ok=True)
            status_path = run / "status.json"
            complete = run / "evaluation_complete.json"
            if complete.exists() and (run / "results.json").exists():
                prior_result = json.loads((run / "results.json").read_text())
                restored_config = {k: v for k, v in prior_result["config"].items() if k != "reference_plan_config"}
                if restored_config != config:
                    raise ValueError(f"Completed run config differs: {run}")
                finished.append(run)
                continue
            if (run / "failure.json").exists(): (run / "failure.json").unlink()
            write_json(status_path, {"phase": "starting", "run_id": run.name})
            env = os.environ.copy()
            env.update({"CUDA_VISIBLE_DEVICES": str(device), "OMP_NUM_THREADS": "2",
                        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
                        "MKL_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
            log = (run / "worker.log").open("a", encoding="utf-8")
            process = subprocess.Popen([sys.executable, "-u", str(worker_path), str(cfg_path)],
                                       env=env, stdout=log, stderr=subprocess.STDOUT)
            active[device] = (process, log, run)
        clear_output(wait=True)
        print("CtrS GPU 실험 진행 상황")
        print(f"실행 완료 {len(finished)}/{len(configs)} | 대기 {len(pending)} | 사용 GPU {gpu_count}")

        # Keep hardware telemetry on one line; never dump multi-line nvidia-smi output.
        try:
            gpu_status = subprocess.run(
                ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=3)
            if gpu_status.returncode == 0:
                gpu_line = " | ".join(part.strip() for part in gpu_status.stdout.splitlines() if part.strip())
                if gpu_line:
                    print("GPU 상태(index, util%, VRAM MiB/total): " + gpu_line)
        except (OSError, subprocess.TimeoutExpired):
            pass

        for device, (process, log, run) in list(active.items()):
            try:
                state = json.loads((run / "status.json").read_text())
            except (OSError, ValueError):
                state = {"phase": "status_update"}
            percent, label = _progress_label(state)
            allocated = float(state.get("allocated_gb", 0.0))
            peak = float(state.get("peak_gb", 0.0))
            print(f"GPU {device} | {run.name} | {percent:5.1f}% | {label} | VRAM {allocated:.1f}/{peak:.1f} GB")

            rc = process.poll()
            if rc is not None:
                log.close()
                del active[device]
                if rc:
                    failure = run / "failure.json"
                    detail = failure.read_text() if failure.exists() else (run / "worker.log").read_text()[-6000:]
                    raise RuntimeError(f"Worker failed: {run}\n{detail}")
                finished.append(run)
        if pending or active:
            time.sleep(5)
finally:
    # Stopping this cell also stops its child training processes.
    for process, log, run in active.values():
        process.terminate()
    for process, log, run in active.values():
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        log.close()
    # A stopped cell remains recoverable: preserve only this suite's files.
    if len(finished) != len(configs):
        recovery = OUT.parent / (OUT.name + "_recovery.zip")
        try:
            with zipfile.ZipFile(recovery, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
                for path in sorted(OUT.rglob("*")):
                    if path.is_file() and path.suffix not in (".tmp", ".pyc") and "__pycache__" not in path.parts:
                        bundle.write(path, arcname=str(Path(OUT.name) / path.relative_to(OUT)))
            print(f"[RECOVERY] {recovery}; Save Version with Output to retain checkpoints", flush=True)
        except OSError as error:
            print(f"[RECOVERY] ZIP unavailable: {error}; suite files remain at {OUT}", flush=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import csv
import re
results = {Path(c["run_dir"]).name: json.loads((Path(c["run_dir"]) / "results.json").read_text())
           for c, _ in configs}
summary_rows = []
for run_id, result in results.items():
    seed = result["config"]["seed"]
    mode = result["config"]["mode"]
    base = results[f"B1_{SENSOR}_B1_k{K}_s{seed}"]["metrics"]
    result["delta_from_same_seed_B1"] = {
        protocol: {key: val - base[protocol]["mean"][key] for key, val in block["mean"].items()}
        for protocol, block in result["metrics"].items()}
    for protocol, block in result["metrics"].items():
        for key, val in block["mean"].items():
            summary_rows.append({"run_id": run_id, "mode": mode, "seed": seed, "best_epoch": result["best_epoch"],
                "protocol": protocol, "metric": key, "mean": val,
                "delta_from_B1": result["delta_from_same_seed_B1"][protocol][key]})

summary_csv = OUT / "comparison_summary.csv"
fieldnames = ["run_id", "mode", "seed", "best_epoch", "protocol", "metric", "mean", "delta_from_B1"]
with summary_csv.open("w", newline="", encoding="utf-8-sig") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

summary_json = OUT / "comparison_summary.json"
write_json(summary_json, {
    run_id: {
        "mode": result["config"]["mode"],
        "seed": result["config"]["seed"],
        "best_epoch": result["best_epoch"],
        "metrics": result["metrics"],
        "delta_from_same_seed_B1": result["delta_from_same_seed_B1"],
    }
    for run_id, result in results.items()
})

# One small plot per protocol/metric keeps differently scaled metrics readable.
plot_dir = OUT / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
for protocol in ("RR", "FR"):
    metrics = sorted({key for result in results.values()
                      for key in result["metrics"].get(protocol, {}).get("mean", {})})
    for metric in metrics:
        labels, values = [], []
        for run_id, result in results.items():
            mean_values = result["metrics"].get(protocol, {}).get("mean", {})
            if metric in mean_values:
                labels.append(run_id)
                values.append(mean_values[metric])
        if not values:
            continue
        fig, ax = plt.subplots(figsize=(max(6, 1.5 * len(labels)), 4))
        ax.bar(labels, values)
        ax.set_title(f"{protocol}: {metric}")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", labelrotation=20)
        fig.tight_layout()
        safe_metric = re.sub(r"[^A-Za-z0-9_.-]+", "_", metric)
        fig.savefig(plot_dir / f"{protocol}_{safe_metric}.png", dpi=140)
        plt.close(fig)

print("실험 및 비교 요약 완료")
print(f"요약 CSV: {summary_csv}")
print(f"요약 JSON: {summary_json}")
print(f"비교 그래프: {plot_dir}")
